# Phase 1 A2d Riemannian Comparator Smoke Plan

Notebook này chuẩn bị bước tiếp theo sau A2/A2b model-smoke: **A2d Riemannian comparator smoke/readiness**.

Phạm vi cố định:

- Chỉ orchestration, audit và artefact planning.
- Không viết logic A2d trực tiếp trong notebook.
- Không tạo claim khoa học từ smoke.
- Nếu runner A2d chưa có trong `src/` và CLI chưa expose lệnh tương ứng, notebook phải ghi blocker rồi dừng.

Done khi notebook ghi được plan artefact hoặc blocker artefact có hash/source rõ ràng trong Drive.
Notebook fix marker: `a2d_prereg_identity_hash_v2_20260420`.



## Cơ sở kỹ thuật và liêm chính khoa học

Nguồn nội bộ phải đối chiếu:

- `docs/01_v55_doc_code_crosswalk.md`: A2d là comparator bắt buộc, Phase 1 smoke chỉ là implementation diagnostics.
- `docs/02_colab_local_runbook.md`: real phases chỉ mở khi Gate 0/1/2/2.5 và exact prereg bundle hợp lệ.
- `notebooks/README.md`: notebook chỉ orchestration; durable scientific logic phải ở `src/` và có tests.
- `V5_5_Technical_Implementation_Spec_vi_complete.docx`: A2b/A2c/A2d là comparator bắt buộc; outer-test subject không được tham gia fit, kể cả alignment object của A2d.
- `V5_5_Execution_Supplement_Implementation_Annex_vi.docx`: threshold registry, comparator inventory, teacher config, control suite phải bị khóa trước substantive runs.

Đối chứng kỹ thuật pyRiemann:

- `pyriemann.tangentspace.TangentSpace.fit()` ước lượng reference matrix; `transform()` chỉ chiếu SPD matrices mới sang tangent space.
- `pyriemann.transfer.TLCenter.fit_transform()` được tài liệu mô tả là dành cho training set; `transform()` dành cho test target domain. Vì vậy A2d runner phải audit rõ fit/transform isolation.

Ranh giới claim:

- A2d smoke không chứng minh decoder efficacy.
- A2d smoke không chứng minh privileged-transfer efficacy.
- A2d smoke không thay thế full Phase 1 comparator suite, controls, calibration và influence package.

In [ ]:
# ============================================================
# Cell 1: Colab bootstrap, token-safe clone/pull, unit tests
# ============================================================
# Purpose:
# - Mount Drive.
# - Clone or update the private repo without printing GitHub token.
# - Run unit tests before any downstream planning.
# - Keep signal extras optional because this notebook does not implement A2d math.

from __future__ import annotations

import base64
import getpass
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive mount unavailable: {exc}')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
INSTALL_SIGNAL_EXTRAS = False


def _redact_cmd(cmd):
    redacted = []
    skip_next = False
    for item in cmd:
        if skip_next:
            redacted.append('<redacted>')
            skip_next = False
            continue
        if item == '-c':
            redacted.append(item)
            skip_next = True
            continue
        redacted.append(item)
    return redacted


def run(cmd, *, cwd=None, env=None, check=True, capture=False):
    print('$', ' '.join(_redact_cmd([str(x) for x in cmd])))
    if not capture:
        return subprocess.run(cmd, cwd=cwd, env=env, check=check)
    result = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
    return result


def git_auth_header(token: str) -> str:
    raw = f'x-access-token:{token}'.encode('utf-8')
    return 'Authorization: Basic ' + base64.b64encode(raw).decode('ascii')

if REPO_DIR.exists():
    run(['git', 'status', '--short', '--branch'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    token = getpass.getpass('Nhập GitHub token cho repo private: ')
    header = git_auth_header(token)
    try:
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    finally:
        token = None
        header = None

os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

NOTEBOOK_FIX_MARKER = 'a2d_prereg_identity_hash_v2_20260420'
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)

env = os.environ.copy()
if INSTALL_SIGNAL_EXTRAS:
    env['INSTALL_SIGNAL_EXTRAS'] = '1'
run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=env)

unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR)
if unit_result.returncode != 0:
    print('Unit tests failed. Không tiếp tục plan A2d cho tới khi test baseline sạch.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed.')

In [ ]:
# ============================================================
# Cell 2: Explicit source-of-truth paths
# ============================================================
# Do not silently follow latest.txt for claim-affecting steps.
# These paths should match the reviewed Colab/Drive artifacts.

from pathlib import Path

PREREG_RUN = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'
PREREG_BUNDLE = PREREG_RUN / 'prereg_bundle.json'
EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
EXPECTED_PREREG_HASH = EXPECTED_LOCKED_PREREG_IDENTITY_HASH  # backward-compatible alias; not raw file SHA256

PHASE1_READINESS_RUN = DRIVE_ROOT / 'artifacts/phase1_readiness/20260419T154005857077Z'
A2_A2B_REVIEWED_RUN = DRIVE_ROOT / 'artifacts/phase1_model_smoke/20260419T172746816598Z'
DATASET_ROOT = DRIVE_ROOT / 'data/ds004752'

A2D_PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_a2d_smoke_plan'
A2D_OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_a2d_smoke'

for label, path in {
    'prereg_bundle': PREREG_BUNDLE,
    'phase1_readiness_run': PHASE1_READINESS_RUN,
    'a2_a2b_reviewed_run': A2_A2B_REVIEWED_RUN,
    'dataset_root': DATASET_ROOT,
}.items():
    print(f'{label}: {path} exists={path.exists()}')

In [ ]:
# ============================================================
# Cell 3: Small helpers for hash, JSON, markdown, subprocess
# ============================================================

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


def now_utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')


def find_summary_file(run_dir: Path, preferred_names: list[str]) -> Path | None:
    for name in preferred_names:
        candidate = run_dir / name
        if candidate.exists():
            return candidate
    matches = sorted(run_dir.glob('*summary*.json'))
    return matches[0] if matches else None

In [ ]:
# ============================================================
# Cell 4: Validate reviewed source chain before planning A2d
# ============================================================
# This cell verifies governance artifacts only. It does not validate model efficacy.
#
# Important hash semantics:
# - EXPECTED_LOCKED_PREREG_IDENTITY_HASH is the locked prereg identity written inside the bundle/source-of-truth.
# - raw_prereg_file_sha256 is the byte-level hash of the JSON file currently on Drive.
# - The raw file hash may differ if the JSON was rewritten, line endings changed, or metadata changed.
# - Raw file SHA256 is recorded for audit, but it is not used as the prereg identity assertion.

missing = [
    str(path)
    for path in [PREREG_BUNDLE, PHASE1_READINESS_RUN, A2_A2B_REVIEWED_RUN, DATASET_ROOT]
    if not path.exists()
]
if missing:
    raise FileNotFoundError('Missing required source paths:\n' + '\n'.join(missing))

bundle = read_json(PREREG_BUNDLE)
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
locked_prereg_hash = (
    bundle.get('prereg_bundle_hash_sha256')
    or bundle.get('bundle_hash_sha256')
    or bundle.get('locked_prereg_bundle_hash_sha256')
)
hash_candidates = {
    'expected_locked_prereg_identity_hash': EXPECTED_LOCKED_PREREG_IDENTITY_HASH,
    'locked_prereg_hash_from_bundle': locked_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_file_sha256,
}
print('Prereg status:', bundle.get('status'))
print('Expected locked prereg identity hash:', EXPECTED_LOCKED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', locked_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
if locked_prereg_hash != EXPECTED_LOCKED_PREREG_IDENTITY_HASH:
    raise AssertionError(
        'Locked prereg identity hash does not match expected reviewed hash. Details:\n'
        + json.dumps(hash_candidates, indent=2, ensure_ascii=False)
    )
if raw_prereg_file_sha256 != EXPECTED_LOCKED_PREREG_IDENTITY_HASH:
    print('NOTE: raw file SHA256 differs from locked prereg identity hash; recorded for audit, not treated as prereg identity failure.')

review_note = A2_A2B_REVIEWED_RUN / 'phase1_a2_a2b_model_smoke_review_note.json'
model_summary = A2_A2B_REVIEWED_RUN / 'phase1_model_smoke_summary.json'
if not review_note.exists():
    raise FileNotFoundError(f'Missing A2/A2b review note: {review_note}')
if not model_summary.exists():
    raise FileNotFoundError(f'Missing A2/A2b model-smoke summary: {model_summary}')

review = read_json(review_note)
summary = read_json(model_summary)
print('A2/A2b review status:', review.get('status'))
print('A2/A2b summary status:', summary.get('status'))
print('A2/A2b claim_ready:', summary.get('claim_ready'))
assert review.get('status') == 'phase1_a2_a2b_model_smoke_review_pass_non_claim'
assert summary.get('claim_ready') is False
assert summary.get('final_eegnet_comparator') is False
assert summary.get('does_not_estimate_privileged_transfer_efficacy') is True

readiness_summary = find_summary_file(PHASE1_READINESS_RUN, [
    'phase1_readiness_summary.json',
    'phase1_decoder_readiness_summary.json',
])
print('Readiness summary candidate:', readiness_summary)

source_chain = {
    'prereg_bundle': {
        'path': str(PREREG_BUNDLE),
        'locked_prereg_hash_sha256': locked_prereg_hash,
        'raw_file_sha256': raw_prereg_file_sha256,
    },
    'phase1_readiness_run': {'path': str(PHASE1_READINESS_RUN), 'summary_candidate': str(readiness_summary) if readiness_summary else None},
    'a2_a2b_reviewed_run': {'path': str(A2_A2B_REVIEWED_RUN), 'review_note_sha256': sha256_file(review_note), 'summary_sha256': sha256_file(model_summary)},
    'dataset_root': str(DATASET_ROOT),
}
source_chain


In [ ]:
# ============================================================
# Cell 5: Hash relevant docs/configs and freeze the planning evidence
# ============================================================
# These hashes are evidence for what governed the A2d plan at notebook runtime.

DOC_AND_CONFIG_PATHS = [
    'docs/01_v55_doc_code_crosswalk.md',
    'docs/02_colab_local_runbook.md',
    'notebooks/README.md',
    'docs/V5_5_Technical_Implementation_Spec_vi_complete.docx',
    'docs/V5_5_Execution_Supplement_Implementation_Annex_vi.docx',
    'docs/V5_5_Master_Artifact_Dossier_Freeze_Prereg_Reporting_Control.docx',
    'docs/blueprint_trien_khai_v1_colab.docx',
    'configs/models/riemannian_a2d.yaml',
    'configs/prereg/prereg_assembly.json',
    'configs/controls/control_suite_spec.yaml',
    'configs/controls/nuisance_block_spec.yaml',
    'configs/eval/metrics.yaml',
    'configs/eval/inference_defaults.yaml',
    'configs/eval/claim_mapping.yaml',
    'configs/gate1/decision_simulation.json',
    'configs/gate2/synthetic_validation.json',
]

doc_config_hashes = {}
for rel in DOC_AND_CONFIG_PATHS:
    path = REPO_DIR / rel
    doc_config_hashes[rel] = {
        'exists': path.exists(),
        'sha256': sha256_file(path) if path.exists() else None,
    }

missing_docs = [rel for rel, info in doc_config_hashes.items() if not info['exists']]
if missing_docs:
    raise FileNotFoundError('Missing docs/configs for A2d plan:\n' + '\n'.join(missing_docs))

print(json.dumps(doc_config_hashes, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Cell 6: A2d runner/config/dependency preflight
# ============================================================
# A2d must be implemented in src/ and exposed through CLI before this notebook can launch it.
# This cell intentionally treats notebook-only implementation as a blocker.

import importlib.util

A2D_COMPARATOR_CONFIG = REPO_DIR / 'configs/models/riemannian_a2d.yaml'
A2D_PHASE_CONFIG = REPO_DIR / 'configs/phase1/a2d_smoke.json'
A2D_MODULE = REPO_DIR / 'src/phase1/a2d_smoke.py'
CLI_PATH = REPO_DIR / 'src/cli.py'

cli_text = CLI_PATH.read_text(encoding='utf-8') if CLI_PATH.exists() else ''
a2d_config_text = A2D_COMPARATOR_CONFIG.read_text(encoding='utf-8') if A2D_COMPARATOR_CONFIG.exists() else ''
a2d_phase_config = read_json(A2D_PHASE_CONFIG) if A2D_PHASE_CONFIG.exists() else {}

pyriemann_spec = importlib.util.find_spec('pyriemann')
numpy_spec = importlib.util.find_spec('numpy')
mne_spec = importlib.util.find_spec('mne')
sklearn_spec = importlib.util.find_spec('sklearn')

runner_backend = a2d_phase_config.get('backend')
pyriemann_required_by_runner = runner_backend == 'pyriemann'

preflight = {
    'status': 'phase1_a2d_riemannian_preflight_checked',
    'a2d_comparator_config': str(A2D_COMPARATOR_CONFIG),
    'a2d_comparator_config_exists': A2D_COMPARATOR_CONFIG.exists(),
    'a2d_comparator_config_placeholder': 'placeholder_until_prereg' in a2d_config_text,
    'a2d_phase_config': str(A2D_PHASE_CONFIG),
    'a2d_phase_config_exists': A2D_PHASE_CONFIG.exists(),
    'a2d_phase_config_backend': runner_backend,
    'a2d_module': str(A2D_MODULE),
    'a2d_module_exists': A2D_MODULE.exists(),
    'cli_has_phase1_a2d_command': 'phase1_a2d_smoke' in cli_text and 'add_parser' in cli_text,
    'cli_has_a2d_smoke_flag': '--a2d-smoke' in cli_text,
    'cli_has_a2d_runner_symbol': 'run_phase1_a2d_smoke' in cli_text,
    'numpy_available': numpy_spec is not None,
    'mne_available': mne_spec is not None,
    'pyriemann_available': pyriemann_spec is not None,
    'pyriemann_required_by_runner': pyriemann_required_by_runner,
    'sklearn_available': sklearn_spec is not None,
    'runtime_warnings': [],
}

preflight['runner_available'] = bool(
    preflight['a2d_module_exists']
    and preflight['a2d_phase_config_exists']
    and (
        preflight['cli_has_phase1_a2d_command']
        or preflight['cli_has_a2d_smoke_flag']
        or preflight['cli_has_a2d_runner_symbol']
    )
)

preflight['blockers'] = []
if preflight['a2d_comparator_config_placeholder']:
    preflight['blockers'].append('configs/models/riemannian_a2d.yaml is still placeholder_until_prereg.')
if not preflight['a2d_module_exists']:
    preflight['blockers'].append('src/phase1/a2d_smoke.py is not implemented.')
if not preflight['a2d_phase_config_exists']:
    preflight['blockers'].append('configs/phase1/a2d_smoke.json is not implemented.')
if not (preflight['cli_has_phase1_a2d_command'] or preflight['cli_has_a2d_smoke_flag'] or preflight['cli_has_a2d_runner_symbol']):
    preflight['blockers'].append('CLI does not expose an A2d smoke/readiness runner.')
if not preflight['numpy_available']:
    preflight['blockers'].append('numpy is not installed; internal A2d smoke backend requires numpy.')
if pyriemann_required_by_runner and not preflight['pyriemann_available']:
    preflight['blockers'].append('pyriemann is required by the configured A2d backend but is not installed.')
if not preflight['mne_available']:
    preflight['runtime_warnings'].append('mne is not installed; real EDF extraction will require INSTALL_SIGNAL_EXTRAS=1 before launch. This is not a blocker for planning/manual hold.')
if not preflight['pyriemann_available'] and not pyriemann_required_by_runner:
    preflight['runtime_warnings'].append('pyriemann is not installed; current A2d smoke backend is internal NumPy and does not require pyriemann.')

print(json.dumps(preflight, indent=2, ensure_ascii=False))


## A2d implementation contract required before launch

The A2d runner must satisfy these minimum rules before this notebook is allowed to launch it:

1. Read the locked prereg bundle and Phase 1 readiness source; reject mismatched hash chains.
2. Use nested LOSO outer folds; never include `outer_test_subject` in train, alignment fit, tangent reference fit, classifier fit, threshold fit, calibration fit, or tuned QC.
3. For pyRiemann-style tangent space, fit reference/alignment objects on training data only, then call transform on the held-out subject.
4. Write per-fold logs proving train/test subject separation and fit/transform object provenance.
5. Write smoke metrics as diagnostics only, with `claim_ready: false`.
6. Write calibration, influence and negative-control artefacts as either executed smoke diagnostics or explicit non-executed shells.
7. Refuse to run if required comparator/config/control files are placeholder or missing.

Expected non-claim artefacts:

- `phase1_a2d_smoke_inputs.json`
- `fold_logs/fold_*.json`
- `a2d_metrics_smoke.json`
- `a2d_alignment_audit.json`
- `a2d_covariance_manifest.json`
- `calibration_a2d_smoke_report.json`
- `negative_controls_a2d_smoke_report.json`
- `influence_a2d_smoke_report.json`
- `phase1_a2d_smoke_summary.json`
- `phase1_a2d_smoke_report.md`

In [ ]:
# ============================================================
# Cell 7: Write A2d plan artifact
# ============================================================
# This records the decision basis and current blocker state without launching training.

run_stamp = now_utc_stamp()
plan_dir = A2D_PLAN_ROOT / run_stamp
plan_dir.mkdir(parents=True, exist_ok=False)

plan = {
    'status': 'phase1_a2d_riemannian_smoke_plan_recorded',
    'created_utc': run_stamp,
    'scope': 'A2d Riemannian comparator smoke/readiness planning only',
    'non_claim_debug_only': True,
    'claim_ready': False,
    'dataset_root': str(DATASET_ROOT),
    'source_chain': source_chain,
    'doc_config_hashes': doc_config_hashes,
    'preflight': preflight,
    'technical_basis': {
        'internal_docs': [
            'V5_5_Technical_Implementation_Spec_vi_complete.docx',
            'V5_5_Execution_Supplement_Implementation_Annex_vi.docx',
            'V5_5_Master_Artifact_Dossier_Freeze_Prereg_Reporting_Control.docx',
            'docs/01_v55_doc_code_crosswalk.md',
            'docs/02_colab_local_runbook.md',
            'notebooks/README.md',
        ],
        'external_primary_docs_checked': [
            'https://pyriemann.readthedocs.io/en/latest/generated/pyriemann.tangentspace.TangentSpace.html',
            'https://pyriemann.readthedocs.io/en/latest/generated/pyriemann.transfer.TLCenter.html',
        ],
    },
    'required_a2d_split_discipline': [
        'outer-test subject excluded from classifier training',
        'outer-test subject excluded from covariance/alignment/tangent reference fitting',
        'outer-test subject excluded from calibration, thresholds and tuned QC',
        'test-time input remains scalp EEG only',
    ],
    'expected_artifacts_after_runner_exists': [
        'phase1_a2d_smoke_inputs.json',
        'fold_logs/fold_*.json',
        'a2d_metrics_smoke.json',
        'a2d_alignment_audit.json',
        'a2d_covariance_manifest.json',
        'calibration_a2d_smoke_report.json',
        'negative_controls_a2d_smoke_report.json',
        'influence_a2d_smoke_report.json',
        'phase1_a2d_smoke_summary.json',
        'phase1_a2d_smoke_report.md',
    ],
    'not_ok_to_claim': [
        'decoder efficacy',
        'privileged-transfer efficacy',
        'A2d final comparator performance',
        'full Phase 1 neural comparator performance',
        'A4 unique privileged value',
    ],
    'next_allowed_steps': [
        'Implement A2d runner in src/phase1 with tests and CLI exposure.',
        'Update configs/phase1/a2d_smoke.json and replace placeholder comparator config only with documented revision/refreeze discipline.',
        'Rerun unit tests locally and in Colab before any manual A2d smoke launch.',
    ],
}

plan_path = plan_dir / 'phase1_a2d_riemannian_smoke_plan.json'
write_json(plan_path, plan)

report_lines = [
    '# Phase 1 A2d Riemannian Smoke Plan',
    '',
    f'- Created UTC: `{run_stamp}`',
    f'- Prereg bundle: `{PREREG_BUNDLE}`',
    f'- Expected locked prereg identity hash: `{EXPECTED_LOCKED_PREREG_IDENTITY_HASH}`',
    f'- Locked prereg hash: `{locked_prereg_hash}`',
    f'- Raw prereg file SHA256: `{raw_prereg_file_sha256}`',
    f'- Dataset root: `{DATASET_ROOT}`',
    f'- Runner available: `{preflight["runner_available"]}`',
    f'- Claim ready: `{plan["claim_ready"]}`',
    '',
    '## Blockers',
]
if preflight['blockers']:
    report_lines.extend([f'- {item}' for item in preflight['blockers']])
else:
    report_lines.append('- None from preflight. Manual launch still requires review acknowledgement.')
report_lines.extend([
    '',
    '## Scientific Limit',
    'This is planning/smoke readiness only. It cannot support decoder efficacy or privileged-transfer efficacy claims.',
])
report_path = plan_dir / 'phase1_a2d_riemannian_smoke_plan.md'
write_text(report_path, '\n'.join(report_lines) + '\n')

latest_path = A2D_PLAN_ROOT / 'latest.txt'
write_text(latest_path, str(plan_dir) + '\n')

print('Plan written:', plan_path)
print('Report written:', report_path)
print(json.dumps({
    'status': plan['status'],
    'plan_dir': str(plan_dir),
    'runner_available': preflight['runner_available'],
    'blockers': preflight['blockers'],
    'claim_ready': plan['claim_ready'],
}, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# Cell 8: Manual launch gate or blocker
# ============================================================
# Default is hold. Do not set RUN_A2D_RIEMANNIAN_SMOKE=True until the plan and runner are reviewed.

RUN_A2D_RIEMANNIAN_SMOKE = False
MANUAL_LAUNCH_ACK = ''
REQUIRED_MANUAL_LAUNCH_ACK = 'I reviewed the A2d plan and understand this is non-claim implementation smoke'

blocker_path = plan_dir / 'phase1_a2d_riemannian_smoke_blocker.json'
manual_hold_path = plan_dir / 'phase1_a2d_riemannian_smoke_manual_hold.json'
launch_review_path = plan_dir / 'phase1_a2d_riemannian_smoke_launch_review.json'

if not preflight['runner_available']:
    blocker = {
        'status': 'phase1_a2d_riemannian_smoke_blocked_runner_unavailable',
        'reason': 'A2d smoke/readiness runner is not available in src/ and CLI.',
        'blockers': preflight['blockers'],
        'required_before_launch': [
            'Implement src/phase1/a2d_smoke.py or equivalent tested module.',
            'Expose an explicit CLI route for A2d smoke/readiness.',
            'Add configs/phase1/a2d_smoke.json with non-claim smoke scope.',
            'Add tests proving outer-test subject is excluded from alignment/reference/classifier/calibration fitting.',
            'Run python -m unittest discover -s tests successfully.',
        ],
        'claim_ready': False,
    }
    write_json(blocker_path, blocker)
    print('BLOCKED:', blocker_path)
    print('NEXT: implement tested A2d runner in src/ before launching from Colab.')
elif not RUN_A2D_RIEMANNIAN_SMOKE:
    hold = {
        'status': 'phase1_a2d_riemannian_smoke_manual_hold',
        'reason': 'Runner appears available, but RUN_A2D_RIEMANNIAN_SMOKE is False by default.',
        'required_manual_action': 'Review plan/source hashes, then set RUN_A2D_RIEMANNIAN_SMOKE=True and provide the exact MANUAL_LAUNCH_ACK string.',
        'required_ack': REQUIRED_MANUAL_LAUNCH_ACK,
        'claim_ready': False,
    }
    write_json(manual_hold_path, hold)
    print('HELD:', manual_hold_path)
elif MANUAL_LAUNCH_ACK != REQUIRED_MANUAL_LAUNCH_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch A2d smoke without explicit non-claim acknowledgement.')
elif not preflight.get('mne_available'):
    raise RuntimeError('A2d real EDF extraction requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS = True in Cell 1, rerun bootstrap/tests, then relaunch intentionally.')
else:
    launch_review = {
        'status': 'manual_launch_approved_for_a2d_riemannian_smoke',
        'approved_flag': RUN_A2D_RIEMANNIAN_SMOKE,
        'ack': MANUAL_LAUNCH_ACK,
        'plan_dir': str(plan_dir),
        'prereg_bundle': str(PREREG_BUNDLE),
        'expected_locked_prereg_identity_hash': EXPECTED_LOCKED_PREREG_IDENTITY_HASH,
        'locked_prereg_hash': locked_prereg_hash,
        'raw_prereg_file_sha256': raw_prereg_file_sha256,
        'readiness_run': str(PHASE1_READINESS_RUN),
        'dataset_root': str(DATASET_ROOT),
        'claim_ready': False,
    }
    write_json(launch_review_path, launch_review)

    # This branch is reserved for the future tested A2d CLI.
    # Keep the command in one place and do not implement model logic here.
    if preflight['cli_has_a2d_smoke_flag']:
        cmd = [
            'python', '-m', 'src.cli', 'phase1_real',
            '--profile', 't4_safe',
            '--config', str(PREREG_BUNDLE),
            '--readiness-run', str(PHASE1_READINESS_RUN),
            '--dataset-root', str(DATASET_ROOT),
            '--output-root', str(A2D_OUTPUT_ROOT),
            '--a2d-smoke',
            '--phase-config', 'configs/phase1/a2d_smoke.json',
            '--max-outer-folds', '2',
        ]
    else:
        raise RuntimeError('Runner preflight changed unexpectedly; no recognized A2d CLI route.')

    run(cmd, cwd=REPO_DIR, capture=True)
    print('A2d smoke command completed. Review generated artifacts before any further run.')

In [ ]:
# ============================================================
# Cell 9: Closeout summary
# ============================================================

print('================ Phase 1 A2d Riemannian Smoke Closeout ================')
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', preflight['runner_available'])
print('Run flag:', RUN_A2D_RIEMANNIAN_SMOKE)
print('Expected locked prereg identity hash:', EXPECTED_LOCKED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', locked_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
print('Blockers:', preflight['blockers'])

if not preflight['runner_available']:
    print('\nBLOCKED: A2d runner/config/CLI is not yet available. This is expected until implementation lands in src/ with tests.')
    print('NEXT: close this notebook after confirming the blocker artifact, then implement the A2d runner locally.')
elif not RUN_A2D_RIEMANNIAN_SMOKE:
    print('\nHELD: Runner appears available, but training was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun with explicit acknowledgement only if appropriate.')
else:
    print('\nCHECK REQUIRED: A2d smoke command was launched. Review fold logs, alignment audit and metrics before continuing.')

print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy or privileged-transfer efficacy.')